# DanmakuCurve：B 站弹幕情绪曲线

贴一个 B 站视频链接，把弹幕标成 **开心 / 惊讶 / 感动 / 愤怒 / 无感**，按视频时间画成曲线。

**怎么用（3 步）**
1. 菜单 **代码执行程序 → 更改运行时类型**，选 **T4 GPU**，保存（只要设一次）。
2. 在下面 ① 里换成你的视频链接。
3. 菜单 **代码执行程序 → 全部运行**。第一次会问"仍然运行"，点它。

图会显示在最下面，PNG 和 CSV 会自动下载。

免费：用的是开源小模型 Qwen2.5-1.5B-Instruct（Apache 2.0），不用 key，不用注册。没有 GPU 时自动改用弹幕词表。
请只用来分析不涉及政治的视频。

In [ ]:
#@title ① 填链接
视频链接 = "https://www.bilibili.com/video/BV1JjZsYHEHE"  #@param {type:"string"}
每段秒数 = 5  #@param {type:"integer"}
标注方式 = "小模型（免费，需 GPU）"  #@param ["小模型（免费，需 GPU）", "词表（最快）", "API（需要 key）"]
模型 = "Qwen/Qwen2.5-1.5B-Instruct"  #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"]

In [ ]:
#@title ② 准备环境（不用改，约 1 分钟）
!pip -q install jieba cnsenti
!apt-get -qq install -y fonts-noto-cjk > /dev/null
import os
WORKDIR = "/content/danmakucurve"
os.makedirs(WORKDIR, exist_ok=True)

open(os.path.join(WORKDIR, 'danmaku.py'), 'w', encoding='utf-8').write(r'''"""抓弹幕 + 解析弹幕 XML。

B 站公开接口，不用登录：
  1. https://api.bilibili.com/x/player/pagelist?bvid=BV...      -> 拿 cid
  2. https://api.bilibili.com/x/v1/dm/list.so?oid=<cid>         -> 弹幕 XML（最多约 1000 条）
"""
import re
import xml.etree.ElementTree as ET

UA = "Mozilla/5.0 (DanmakuCurve research script)"


def bvid_from_url(url_or_bvid):
    m = re.search(r"(BV[0-9A-Za-z]{10})", url_or_bvid)
    if not m:
        raise ValueError(f"没找到 BV 号: {url_or_bvid}")
    return m.group(1)


def _decode(raw):
    """把接口返回的字节变成文字。不看响应头，按内容逐个试：
    未压缩、gzip、zlib、不带头的 deflate、brotli。哪个能解成 UTF-8 就用哪个。"""
    import gzip
    import zlib

    tries = [
        lambda b: b,
        gzip.decompress,
        zlib.decompress,
        lambda b: zlib.decompress(b, -zlib.MAX_WBITS),
    ]
    try:
        import brotli
        tries.append(brotli.decompress)
    except ImportError:
        pass
    for f in tries:
        try:
            return f(raw).decode("utf-8")
        except Exception:
            continue
    raise ValueError(f"接口返回的内容解不开（开头几个字节: {raw[:8]!r}）")


def _get_text(url, params, timeout):
    """B 站弹幕接口常返回压缩过的内容，而且响应头不一定写对，所以拿原始字节自己解。"""
    import requests

    headers = {
        "User-Agent": UA,
        "Referer": "https://www.bilibili.com/",
        "Accept-Encoding": "gzip, deflate",
    }
    r = requests.get(url, params=params, headers=headers, timeout=timeout, stream=True)
    r.raise_for_status()
    return _decode(r.raw.read(decode_content=False))


def fetch_xml(url_or_bvid, page=1, timeout=15):
    """返回 (title, xml_text)。需要能连上 api.bilibili.com。"""
    import json

    bvid = bvid_from_url(url_or_bvid)
    data = json.loads(_get_text("https://api.bilibili.com/x/player/pagelist", {"bvid": bvid}, timeout))
    if data.get("code") != 0:
        raise RuntimeError(f"B 站返回错误: {data.get('message')}")
    part = data["data"][page - 1]
    cid = part["cid"]
    title = part.get("part") or bvid
    try:  # 视频标题，拿不到就用分 P 名
        view = json.loads(_get_text("https://api.bilibili.com/x/web-interface/view", {"bvid": bvid}, timeout))
        title = view["data"]["title"] or title
    except Exception:
        pass
    return title, _get_text("https://api.bilibili.com/x/v1/dm/list.so", {"oid": cid}, timeout)


def parse_xml(xml_text):
    """弹幕 XML -> [(视频秒数, 文本)]，按时间排序。"""
    root = ET.fromstring(xml_text.encode("utf-8") if isinstance(xml_text, str) else xml_text)
    rows = []
    for d in root.iter("d"):
        p = d.get("p", "")
        text = (d.text or "").strip()
        if not p or not text:
            continue
        rows.append((float(p.split(",")[0]), text))
    rows.sort(key=lambda r: r[0])
    return rows
''')
open(os.path.join(WORKDIR, 'label.py'), 'w', encoding='utf-8').write(r'''"""给每条弹幕标 5 类情绪：开心、惊讶、感动、愤怒、无感。

免费、不用注册、不用 key：
  第 1 层：弹幕黑话词表（下面的 RULES，自己写的，随时可改）
  第 2 层：开源中文情绪词典 cnsenti（大连理工 DUTIR 情感词汇本体，7 类）
  都没命中 -> 无感

以后想换成 LLM，只要写一个同样签名的 label(text) -> str 函数替换即可。
"""
import re

LABELS = ["开心", "惊讶", "感动", "愤怒", "无感"]

# 同分时谁优先：越少见、越有信息量的排前面
PRIORITY = ["感动", "愤怒", "惊讶", "开心"]

# 每类：正则列表。命中一个记 1 分。
# 第二轮（全片 1200 条）加了：玩梗/角色扮演、比例错乱（"猫怎么这么大"）、夸人、emoji。
RULES = {
    "开心": [
        r"哈{2,}", r"h{3,}", r"2333*", r"笑死", r"笑[哭疯了]", r"好笑", r"乐死", r"乐了", r"^草+$",
        r"xswl", r"绷不住", r"没绷住", r"难绷", r"笑(得|的|到)?[^，。]{0,6}(喘|不行|炸|抽|死|疯|麻|裂|喷)", r"气笑", r"憋不住|憋下去", r"神人", r"蚌埠住", r"救命", r"嘿嘿", r"嘻嘻", r"啦啦啦",
        r"可爱", r"好乖", r"喜欢", r"爱了", r"爱上", r"好玩", r"有意思", r"有趣", r"开心", r"快乐",
        r"幸福", r"羡慕", r"好棒", r"太棒", r"不错", r"好看", r"甜", r"嗑", r"doge", r"狗头", r"滑稽",
        r"好耶", r"舒服", r"治愈", r"酷", r"想去", r"想住|也想住|能住", r"也能进", r"宠老婆",
        r"想当你的儿子|还缺儿子", r"好家伙",
        # 玩梗：书名号标题、"进击的…"、"道理我都懂"、给猫/孩子配台词、自嘲括号
        r"^《[^》]+》$", r"进击的", r"道理我都懂", r"既视感", r"私闯民宅", r"怪兽", r"巨兽", r"入侵",
        r"人类幼崽", r"^(?!原曲|原唱|bgm|BGM|出处|来源|省流|前面的|弹幕)[\u4e00-\u9fff]{1,3}\s*[:：∶]", r"[（(](bushi|确信|恳求|狗头|滑稽|doge)",
        r"[😂🤣😆😄😁😊🥰😍❤💕👍]",
    ],
    "惊讶": [
        r"卧槽", r"我去(?=[，,！!。？?\s]|$)", r"我靠", r"握草", r"我[勒嘞]个去", r"我[滴的]天", r"天[哪啊呐]", r"我的妈",
        r"居然", r"竟然", r"真[的能]?能", r"没想到", r"原来", r"震惊", r"离谱", r"不会吧",
        r"啊这", r"^啊[？?]", r"^[？?]+$", r"吓死|吓我",
        # 夸人（5 类里没有"佩服"，算惊讶）
        r"牛", r"\bnb\b", r"厉害|力害", r"太强|超强", r"强强", r"太顶", r"神了", r"天才|人才",
        r"创意", r"精致", r"手搓", r"666+", r"^\D{0,3}6+[!！]?$", r"边形", r"全能", r"天花板", r"豪华", r"细节",
        # 比例错乱：房子太小，看起来像巨猫
        r"这么大|太大了|巨型|巨猫", r"比例", r"分不清|搞不清|没概念", r"参照系", r"晕", r"恍惚",
        r"以为", r"第一反应", r"起猛了",
        r"有水|通水|通电|能用|能动",
        r"[？?]{2,}", r"[！!]{3,}", r"[？?][！!]|[！!][？?]",
        r"[😱😮😲🤯😳]",
    ],
    "感动": [
        r"泪目", r"哭了", r"想哭", r"感动", r"破防", r"呜+", r"眼泪", r"鼻子[一好]?酸",
        r"暖心", r"温暖", r"致敬", r"心疼", r"爷青回", r"怀念", r"回忆", r"加油",
        r"谢谢", r"感谢", r"最幸福", r"梦开始的地方|梦开始地方", r"泪",
        r"╥﹏╥|T_T|QAQ", r"[😭🥺]",
        # 第四轮：善意/功德类（"都是好心人""功德无量""你真好"）
        r"好心人|好人|功德|善良|你真好|善举",
    ],
    "愤怒": [
        r"气死", r"生气", r"恶心", r"^滚", r"垃圾", r"傻[逼b]", r"\bsb\b", r"什么玩意",
        r"无语", r"烦死", r"讨厌", r"有病", r"吐了", r"恶臭", r"差评", r"退钱", r"举报",
        r"气人", r"服了", r"[😡🤬💢]",
        # 第四轮：嫌弃/数落（"这鹿没素质""浪费水！可耻！"）；"不浪费""别浪费"不算
        r"没素质|素质[真太好]?[低差]|缺德|可耻|这么坏|真坏|太坏|坏[比b]",
        r"(?<![不别没免])浪费",
    ],
}
# 刷屏广告直接算无感（这个视频里有一批"炒股小说"广告弹幕）
# 广告和模板刷屏（正在看、弹幕飞过等）一律判无感
SPAM = re.compile(r"藩茄|富豪股神|全款购|^(正在看|正在刷弹幕|弹幕飞过|学到了|打卡|前排)$")
# 笑类说法：小模型（1.5B）认不稳，这些直接标开心，不交给模型。
LAUGH = re.compile(r"没绷住|绷不住|难绷|憋(?:不住|笑|下去)|气笑|笑死|笑不行|笑[得的]?喘|笑炸|笑疯|哈{2,}|2333|hhh", re.I)
_COMPILED = {k: [re.compile(p, re.I) for p in v] for k, v in RULES.items()}

# cnsenti 的 7 类 -> 我们的 5 类。
# 只用 乐/怒/惧：在样例弹幕上试过，"好"把"确实"标成开心，"恶"/"哀"/"惊"也大多标错，所以不用。
DUTIR_MAP = {"乐": "开心", "怒": "愤怒", "惧": "惊讶"}

_emotion = None


def _dutir(text):
    """cnsenti 词典打分；没装就跳过。"""
    global _emotion
    if _emotion is False:
        return None
    if _emotion is None:
        try:
            import logging
            import jieba
            jieba.setLogLevel(logging.WARNING)
            from cnsenti import Emotion
            _emotion = Emotion()
        except Exception:
            _emotion = False
            return None
    counts = _emotion.emotion_count(text)
    scores = {}
    for k, ours in DUTIR_MAP.items():
        scores[ours] = scores.get(ours, 0) + counts.get(k, 0)
    return _pick(scores)


def _pick(scores):
    best = max(scores.values(), default=0)
    if best <= 0:
        return None
    for lab in PRIORITY:
        if scores.get(lab, 0) == best:
            return lab
    return None


def label(text):
    if SPAM.search(text):
        return "无感"
    scores = {lab: sum(1 for p in pats if p.search(text)) for lab, pats in _COMPILED.items()}
    # 带“笑”的句子（如“什么玩意，给我笑炸了”）是在笑，不算愤怒
    if "笑" in text and scores.get("开心"):
        scores["愤怒"] = 0
    # 带玩笑标记（bushi、doge、狗头）的不算感动，如“功德+3（bushi）”
    if re.search(r"bushi|doge|狗头", text, re.I):
        scores["感动"] = 0
    return _pick(scores) or _dutir(text) or "无感"


def label_all(rows):
    """[(t, text)] -> [(t, text, 情绪)]"""
    return [(t, txt, label(txt)) for t, txt in rows]
''')
open(os.path.join(WORKDIR, 'curve.py'), 'w', encoding='utf-8').write(r'''"""按视频时间分段，统计每段各情绪条数，画成 PNG。"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

from label import LABELS

# 紫色为主色（开心），其余三色用来区分；已检查色盲可分辨
COLORS = {"开心": "#7c3aed", "惊讶": "#0891b2", "感动": "#db2777",
          "愤怒": "#d97706", "无感": "#a1a1aa"}
BAR = "#ede9fe"  # 每段弹幕总数的底色（浅紫）

_CJK = ["WenQuanYi Zen Hei", "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans SC", "PingFang SC",
        "Microsoft YaHei", "SimHei", "Heiti SC", "Source Han Sans SC"]


def _set_cjk_font():
    # Colab 装完 fonts-noto-cjk 后，matplotlib 还不认识，手动登记一下
    import glob
    for path in glob.glob("/usr/share/fonts/**/*CJK*", recursive=True):
        try:
            font_manager.fontManager.addfont(path)
        except Exception:
            pass
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in _CJK:
        if name in have:
            # 英文和数字用 DejaVu Sans（matplotlib 自带），中文字符自动回退到中文字体
            plt.rcParams["font.family"] = ["DejaVu Sans", name]
            plt.rcParams["axes.unicode_minus"] = False
            return True
    return False


def bin_counts(labeled, bin_sec=10, duration=None):
    """-> (起点秒数列表, {情绪: [每段条数]})"""
    end = duration if duration else (max(t for t, _, _ in labeled) if labeled else 0)
    n = int(end // bin_sec) + 1
    counts = {lab: [0] * n for lab in LABELS}
    for t, _, lab in labeled:
        i = min(int(t // bin_sec), n - 1)
        counts[lab][i] += 1
    return [i * bin_sec for i in range(n)], counts


def plot(labeled, out_png, title="", bin_sec=10, show_neutral=False):
    starts, counts = bin_counts(labeled, bin_sec)
    _set_cjk_font()
    fig, ax = plt.subplots(figsize=(11, 4.5), dpi=150)
    labs = [l for l in LABELS if show_neutral or l != "无感"]
    for lab in labs:
        ax.plot(starts, counts[lab], marker="o", ms=3, lw=2, color=COLORS[lab], label=lab)
    total = [sum(counts[l][i] for l in LABELS) for i in range(len(starts))]
    ax.bar(starts, total, width=bin_sec * 0.9, align="edge", color=BAR,
           label="全部弹幕", zorder=0)
    ax.set_xlabel(f"视频时间（秒，每 {bin_sec} 秒一段）")
    ax.set_ylabel("弹幕条数")
    ax.set_title(title or "弹幕情绪曲线")
    ax.legend(ncol=6, loc="upper right", frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(out_png)
    plt.close(fig)
    return out_png
''')
open(os.path.join(WORKDIR, 'llm_label.py'), 'w', encoding='utf-8').write(r'''"""用免费开源小模型（默认 Qwen2.5-1.5B-Instruct，Apache 2.0）给弹幕标 5 类情绪。

不用 key、不用注册。需要 GPU（Colab 免费 T4 就够）。
做法：让模型读一条弹幕，只比较 5 个标签第一个字的概率，挑最大的，
所以输出一定是 5 类之一，不会胡写。
广告和模板刷屏先判成无感；没绷住、难绷、笑死这类笑的说法先判成开心（词表优先），都不进模型。
"""
from label import LABELS, LAUGH, SPAM

DEFAULT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

SYSTEM = (
    "你是 B 站弹幕情绪标注员。每次给你一条弹幕，只回答一个词：开心、惊讶、感动、愤怒、无感。\n"
    "开心：好笑、玩梗、可爱、喜欢、调侃。\n"
    "惊讶：意外、震惊、佩服、没想到。\n"
    "感动：泪目、暖心、感谢、致敬、心疼。\n"
    "愤怒：生气、嫌弃、骂人、数落。\n"
    "无感：只是陈述、提问、刷屏，看不出情绪。\n"
    "注意 B 站说法：没绷住、绷不住、难绷、憋不住、憋笑、气笑了、笑死、笑喘不上气、2333、hhh 都是在笑，算开心，不是惊讶。\n"
    "例子：\n"
    "弹幕：没绷住，上来就溜达了 → 开心\n"
    "弹幕：气笑了 → 开心\n"
    "弹幕：给我笑的喘不上气了哥 → 开心\n"
    "弹幕：居然是他？？ → 惊讶\n"
    "弹幕：今天下雨了 → 无感"
)


def _fixed(texts):
    """不用问模型的弹幕：刷屏 -> 无感，笑类说法 -> 开心。"""
    out = {}
    for t in texts:
        if SPAM.search(t):
            out[t] = "无感"
        elif LAUGH.search(t):
            out[t] = "开心"
    return out


class LLMLabeler:
    def __init__(self, model_name=DEFAULT_MODEL, device=None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(self.device).eval()
        # 每个标签的第一个 token；5 个必须互不相同
        self.first_ids = [self.tok.encode(lab, add_special_tokens=False)[0] for lab in LABELS]
        if len(set(self.first_ids)) != len(LABELS):
            raise RuntimeError("这个模型的分词把两个标签开头切成一样了，换个模型试试")

    def _prompt(self, text):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"弹幕：{text}"}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def label_texts(self, texts, batch_size=32, progress=None):
        """[文本] -> [情绪]。重复的弹幕只算一次。"""
        out = _fixed(texts)
        uniq = [t for t in dict.fromkeys(texts) if t not in out]
        ids = self.torch.tensor(self.first_ids, device=self.device)
        for i in range(0, len(uniq), batch_size):
            chunk = uniq[i:i + batch_size]
            enc = self.tok([self._prompt(t) for t in chunk], return_tensors="pt", padding=True).to(self.device)
            with self.torch.no_grad():
                logits = self.model(**enc).logits[:, -1, :]  # 左侧补齐，最后一位就是下一个字
            best = logits.index_select(1, ids).argmax(dim=1).tolist()
            for t, b in zip(chunk, best):
                out[t] = LABELS[b]
            if progress:
                progress(min(i + batch_size, len(uniq)), len(uniq))
        return [out.get(t, "无感") for t in texts]


def label_all_llm(rows, labeler, **kw):
    """[(t, text)] -> [(t, text, 情绪)]"""
    labs = labeler.label_texts([txt for _, txt in rows], **kw)
    return [(t, txt, lab) for (t, txt), lab in zip(rows, labs)]


def label_all_api(rows, api_key, base_url, model, batch_size=40):
    """可选：用 OpenAI 兼容接口（Together、Z.ai 等）标注。key 从 Colab Secrets 读，不写进代码。"""
    import re

    import requests

    texts = [txt for _, txt in rows]
    out = _fixed(texts)
    uniq = [t for t in dict.fromkeys(texts) if t not in out]
    for i in range(0, len(uniq), batch_size):
        chunk = uniq[i:i + batch_size]
        listing = "\n".join(f"{j + 1}. {t}" for j, t in enumerate(chunk))
        body = {
            "model": model,
            "temperature": 0,
            "messages": [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"下面有 {len(chunk)} 条弹幕。按顺序每行只写一个情绪词，共 {len(chunk)} 行，不要编号。\n{listing}"},
            ],
        }
        r = requests.post(base_url.rstrip("/") + "/chat/completions", json=body, timeout=120,
                          headers={"Authorization": f"Bearer {api_key}"})
        r.raise_for_status()
        lines = r.json()["choices"][0]["message"]["content"].strip().splitlines()
        for t, line in zip(chunk, lines):
            m = re.search("|".join(LABELS), line)
            out[t] = m.group(0) if m else "无感"
    return [(t, txt, out.get(txt, "无感")) for t, txt in rows]
''')
print("准备好了")

In [ ]:
#@title （可选）API 设置：只有 ① 里选了「API」才会用
# key 不要写在这里。点左边栏的钥匙图标（密钥 / Secrets），新建一个叫 LLM_API_KEY 的密钥，把 key 填进去，并打开"笔记本访问权限"。
API地址 = "https://api.together.xyz/v1"  #@param ["https://api.together.xyz/v1", "https://api.z.ai/api/paas/v4"]
API模型 = "Qwen/Qwen2.5-7B-Instruct-Turbo"  #@param {type:"string"}

In [ ]:
#@title ③ 跑起来
import collections
import os
import sys

sys.path.insert(0, WORKDIR)
from IPython.display import Image, display

from curve import plot
from danmaku import bvid_from_url, fetch_xml, parse_xml
from label import LABELS, label_all

try:
    bv = bvid_from_url(视频链接)
except ValueError:
    bv = "video"

# 1. 抓弹幕；抓不到就让你上传 XML
try:
    标题, xml_text = fetch_xml(视频链接)
    print("抓到了：", 标题)
except Exception as e:
    print(f"直接抓不到弹幕（{type(e).__name__}: {e}）")
    print("可以手动存一份弹幕 XML 再上传：")
    print(f"  1) 浏览器打开 https://api.bilibili.com/x/player/pagelist?bvid={bv} ，找到 cid 后面的数字")
    print("  2) 打开 https://api.bilibili.com/x/v1/dm/list.so?oid=那个数字 ，另存为 .xml")
    print("  3) 点下面的「选择文件」上传")
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    xml_text, 标题 = up[name].decode("utf-8"), os.path.splitext(name)[0]

rows = parse_xml(xml_text)
print(len(rows), "条弹幕")

# 2. 标情绪
用了 = "词表"
if 标注方式.startswith("小模型"):
    import torch
    if torch.cuda.is_available():
        try:
            from llm_label import LLMLabeler, label_all_llm
            print("下载并加载模型（第一次约 1–2 分钟）……")
            labeler = LLMLabeler(模型)
            labeled = label_all_llm(rows, labeler, progress=lambda a, b: print(f"\r标注 {a}/{b}", end=""))
            print()
            用了 = 模型.split("/")[-1] + "+笑类词表"
        except Exception as e:
            print(f"\n小模型出错（{type(e).__name__}: {e}），改用词表。")
            labeled = label_all(rows)
    else:
        print("现在没有 GPU，先用词表。想用小模型：代码执行程序 → 更改运行时类型 → T4 GPU，再全部运行。")
        labeled = label_all(rows)
elif 标注方式.startswith("API"):
    from google.colab import userdata
    from llm_label import label_all_api
    try:
        key = userdata.get("LLM_API_KEY")
    except Exception:
        raise SystemExit("没找到密钥 LLM_API_KEY。左边栏钥匙图标里新建它，并打开笔记本访问权限。")
    labeled = label_all_api(rows, key, API地址, API模型)
    用了 = API模型
else:
    labeled = label_all(rows)

# 3. 对比词表、画图、存文件
词表 = collections.Counter(l for _, _, l in label_all(rows))
这次 = collections.Counter(l for _, _, l in labeled)
print(f"各情绪条数（{用了} / 词表）：")
for lab in LABELS:
    print(f"  {lab}: {这次[lab]} / {词表[lab]}")

png, csv_path = f"curve_{bv}.png", f"labels_{bv}.csv"
import csv
with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["秒", "弹幕", "情绪"])
    w.writerows((f"{t:.1f}", txt, lab) for t, txt, lab in labeled)
plot(labeled, png, title=f"《{标题}》{len(rows)} 条弹幕（{用了}）", bin_sec=每段秒数)
display(Image(png))
try:
    from google.colab import files
    files.download(png)
    files.download(csv_path)
except Exception:
    print("已保存：", png, csv_path)